# 03 — Baseline: Temporal Forecasting with a Simple CNN

This notebook provides a **minimal ML baseline** for the temporal forecasting benchmark task: predict the next velocity snapshot from previous ones.

This is intentionally simple — the goal is to provide a starting point for benchmarking more advanced architectures (FNO, U-Net, transformers, PINNs, etc.).

**Task**: Given velocity field at time *t*, predict velocity at *t+1*  
**Metric**: Relative L2 error  
**Paper**: [Khojasteh et al. (2021)](https://doi.org/10.1016/j.dib.2021.107725)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from cylinderwake import CylinderWake3900

%matplotlib inline
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Create Forecasting Dataset

Wrap the Eulerian dataset into input-output pairs: (snapshot_t) → (snapshot_{t+1}).

In [ ]:
class ForecastingDataset(Dataset):
    """Wraps CylinderWake3900 for next-step prediction."""
    
    def __init__(self, base_dataset, z_slice=None):
        self.ds = base_dataset
        self.z_slice = z_slice  # Use 2D slice for faster training
    
    def __len__(self):
        return len(self.ds) - 1
    
    def __getitem__(self, idx):
        current = self.ds[idx]
        next_step = self.ds[idx + 1]
        
        x = current['velocity']   # (3, Nx, Ny, Nz)
        y = next_step['velocity']
        
        if self.z_slice is not None:
            x = x[:, :, :, self.z_slice]  # (3, Nx, Ny)
            y = y[:, :, :, self.z_slice]
        
        return x, y


# Load data with canonical splits
train_base = CylinderWake3900('eulerian', 'near', split='train')
val_base   = CylinderWake3900('eulerian', 'near', split='val')
test_base  = CylinderWake3900('eulerian', 'near', split='test')

# Use z mid-plane for fast prototyping
sample = train_base[0]
nz = sample['velocity'].shape[-1]
z_mid = nz // 2

train_ds = ForecastingDataset(train_base, z_slice=z_mid)
val_ds   = ForecastingDataset(val_base, z_slice=z_mid)
test_ds  = ForecastingDataset(test_base, z_slice=z_mid)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')

x, y = train_ds[0]
print(f'Input shape:  {x.shape}')
print(f'Target shape: {y.shape}')

## 2. Simple CNN Baseline

In [ ]:
class SimpleCNN(nn.Module):
    """Minimal CNN for next-step velocity prediction."""
    
    def __init__(self, in_channels=3, out_channels=3, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, out_channels, 3, padding=1),
        )
        # Residual: predict the *change* from t to t+1
    
    def forward(self, x):
        return x + self.net(x)  # residual connection


model = SimpleCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

## 3. Training Loop

In [ ]:
def relative_l2_error(pred, target):
    """Relative L2 error: ||pred - target|| / ||target||"""
    return torch.norm(pred - target) / torch.norm(target)


train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=4)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

n_epochs = 50
train_losses = []
val_losses = []

for epoch in range(n_epochs):
    # Train
    model.train()
    epoch_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = relative_l2_error(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))
    
    # Validate
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            val_loss += relative_l2_error(pred, y).item()
    val_losses.append(val_loss / len(val_loader))
    scheduler.step(val_losses[-1])
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | Train L2: {train_losses[-1]:.4f} | Val L2: {val_losses[-1]:.4f}')

## 4. Training Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Relative L2 Error')
plt.title('Temporal Forecasting — Simple CNN Baseline')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Test Set Evaluation

In [ ]:
test_loader = DataLoader(test_ds, batch_size=1)

model.eval()
test_errors = []
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        test_errors.append(relative_l2_error(pred, y).item())

print(f'\n══════════════════════════════════════════')
print(f'  Test Relative L2 Error: {np.mean(test_errors):.4f} ± {np.std(test_errors):.4f}')
print(f'══════════════════════════════════════════')
print(f'\nThis is a minimal CNN baseline. Improve upon it with:')
print(f'  - FNO (Fourier Neural Operator)')
print(f'  - U-Net / ResNet architectures')
print(f'  - Vision Transformers')
print(f'  - Physics-informed losses')
print(f'  - Multi-step rollout training')

## 6. Qualitative Comparison

In [ ]:
# Pick a test sample
x, y_true = test_ds[0]
x_dev = x.unsqueeze(0).to(device)

with torch.no_grad():
    y_pred = model(x_dev).cpu().squeeze(0)

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
titles_row = ['Input (t)', 'Ground truth (t+1)', 'Prediction (t+1)']
comp_labels = ['$u_x$', '$u_y$', '$u_z$']

for i in range(3):  # components
    vmin = min(x[i].min(), y_true[i].min())
    vmax = max(x[i].max(), y_true[i].max())
    
    for j, (data, title) in enumerate(zip([x, y_true, y_pred], titles_row)):
        im = axes[i, j].imshow(data[i].T, origin='lower', cmap='RdBu_r',
                               aspect='auto', vmin=vmin, vmax=vmax)
        axes[i, j].set_title(f'{comp_labels[i]} — {title}')
        plt.colorbar(im, ax=axes[i, j], shrink=0.7)

plt.suptitle('Temporal Forecasting: CNN Baseline', fontsize=14)
plt.tight_layout()
plt.savefig('forecasting_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Benchmark Results Table

| Method | Rel. L2 Error | Parameters | Notes |
|--------|:------------:|:----------:|-------|
| **Simple CNN (this notebook)** | TBD | ~50k | Residual, 4 layers |
| FNO | — | — | *Your results here* |
| U-Net | — | — | *Your results here* |
| PINN | — | — | *Your results here* |

**We welcome contributions!** Submit your results via a Pull Request to the [benchmark table](https://github.com/Ali-Rahimi-Khojasteh/cylinder-wake-re3900).